# Solar Filament Segmentation, notebook 02: U-Net (PyTorch)

COSC 50523 Deep Learning (TCU), course project. Dataset: MAGFiLO (Kaggle competition `filament-segmentation-2026`).

**Task:** predict a binary mask of solar filament pixels for each full-disk solar image, with masks drawn from the COCO polygon annotations.

**Configuration used in this notebook** (read from the config, loss and training cells; `EPOCHS`, `BATCH`, `ENCODER_WEIGHTS`, `DATA_DIR` can be overridden by environment variables):

| Setting | Value |
|---|---|
| `IMG_SIZE` | 512 (images and masks resized to 512 x 512) |
| Model | `smp.Unet`, `ENCODER = "resnet34"`, `in_channels=1`, `classes=1` |
| `ENCODER_WEIGHTS` | `"imagenet"` (default) |
| `EPOCHS` | 15 (default) |
| `BATCH` | 8 (default) |
| `LR` | 3e-4, Adam; mixed precision on GPU |
| Loss | BCE with `pos_weight = sqrt((1 - p) / p)` (p = train filament fraction) + soft Dice |
| Augmentation (train only) | random flips, 90-degree rotations, brightness/contrast jitter |
| Model selection | epoch with best validation Dice at threshold 0.5 |
| `SPLIT` | (0.70, 0.15, 0.15) |
| `SPLIT_BY_DAY` | True |
| `SEED` | 42 |
| `CATEGORY_IDS` | None (all annotation categories) |
| Threshold search on val | probability 0.05 to 0.95 (19 values) |

**Split rule:** 70 / 15 / 15 (train / val / test) by whole image *and* by day: the day is the first 8 characters of the file name (YYYYMMDD), and all images from one day go to the same split.

**Metrics:** pooled IoU, Dice, precision and recall over all pixels of the held-out images (TP / FP / FN summed across the split). The decision threshold is tuned on validation (max Dice); the test split is scored once.

**Data is NOT included in this repository.** Download it from Kaggle (competition "Solar Filament Segmentation Challenge 2026", slug `filament-segmentation-2026`) after joining the competition and accepting its rules (otherwise the download returns 403). The download cell below asks for the Kaggle token with `getpass` at runtime; never paste a token into a cell or commit `kaggle.json`.

**Attribution:** the U-Net is built with the `segmentation_models_pytorch` library and, unless `ENCODER_WEIGHTS` is set otherwise, starts from ImageNet-pretrained ResNet34 encoder weights downloaded by that library. The data loading, split, loss, training loop and metrics are the team's own code.

# Solar Filament Segmentation: U-Net (PyTorch)

Second model for the project. The HW2 logistic regression is a **single neuron** applied to one pixel's neighborhood.
A U-Net is a **deep convolutional network** that takes a whole image and outputs a whole mask at once, so it can learn
long thin shapes instead of just local intensity.

Pipeline: load images + draw masks from the COCO JSON -> split by day -> augment -> train U-Net with BCE + Dice loss ->
pick best epoch on validation Dice -> tune threshold on validation -> score test images with the same IoU / Dice /
precision / recall as the logistic regression.

**Use a GPU:** Runtime -> Change runtime type -> T4 GPU. On CPU this will be extremely slow.

## 1 - Install, config, imports

In [ ]:
import os, sys, glob, json, random, time
if "google.colab" in sys.modules:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

SEED = 42
IMG_SIZE = 512              # must be divisible by 32. 256 is faster but may erase thin filaments
ENCODER = "resnet34"        # U-Net encoder backbone
ENCODER_WEIGHTS = os.environ.get("ENCODER_WEIGHTS", "imagenet")  # "imagenet" = pretrained; set to None to train from scratch
if ENCODER_WEIGHTS == "None": ENCODER_WEIGHTS = None
EPOCHS = int(os.environ.get("EPOCHS", 15))
BATCH = int(os.environ.get("BATCH", 8))
LR = 3e-4
SPLIT = (0.70, 0.15, 0.15)
SPLIT_BY_DAY = True
CATEGORY_IDS = None         # None = all annotation categories; or e.g. [2] after reading the category printout

DATA_DIR = os.environ.get("DATA_DIR", "/content/filament-segmentation-2026")
_ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
ANN_PATH = _ann[0] if _ann else None
IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images") if ANN_PATH else None

assert IMG_SIZE % 32 == 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| annotations:", ANN_PATH)
if device == "cpu": print("WARNING: no GPU found, training will be very slow")

## 2 - Get the data (Colab only)
Join the competition on Kaggle first. Paste your Kaggle API token when prompted (or leave empty to upload a legacy
`kaggle.json`). Skip this cell if the data is already in `DATA_DIR`, then re-run the config cell so the paths are found.

In [ ]:
if "google.colab" in sys.modules and not os.path.exists(DATA_DIR):
    import getpass
    os.system("pip install -q --upgrade kaggle")
    tok = getpass.getpass("Paste Kaggle API token (leave empty to upload kaggle.json instead): ").strip()
    if tok:
        os.environ["KAGGLE_API_TOKEN"] = tok
    else:
        from google.colab import files
        files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.system("mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
    os.system("kaggle competitions download -c filament-segmentation-2026 -p /content/dl")
    os.system(f"unzip -q -o /content/dl/*.zip -d {DATA_DIR}")
    _ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
    ANN_PATH = _ann[0]; IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images")
print(os.path.exists(DATA_DIR), ANN_PATH)

## 3 - Inspect annotations

In [ ]:
A = json.load(open(ANN_PATH))
print("images:", len(A["images"]), " annotations:", len(A["annotations"]))
cnt = Counter(a["category_id"] for a in A["annotations"])
for c in A["categories"]:
    print(c, "-> annotations:", cnt.get(c["id"], 0))

In [ ]:
# Categories found in the COCO JSON and the selection used for the masks
for c in sorted(A["categories"], key=lambda c: c["id"]):
    print(f"category id={c['id']}  name={c['name']!r}  supercategory={c.get('supercategory')!r}")
print("CATEGORY_IDS =", CATEGORY_IDS, " (None means all categories)")

TEAM TO WRITE: which categories correspond to filaments, and whether CATEGORY_IDS should be restricted.

## 4 - Load images and draw masks from the polygons
Same loader as the logistic regression notebook. Images are grayscale scaled to 0-1; masks come from the COCO polygons.

In [ ]:
def load_pairs(ann_path, img_dir, size=IMG_SIZE, category_ids=CATEGORY_IDS, limit=None):
    A = json.load(open(ann_path))
    by_img = defaultdict(list)
    for a in A["annotations"]:
        if category_ids is None or a["category_id"] in category_ids:
            by_img[a["image_id"]].append(a)
    X, Y, names, skipped_rle = [], [], [], 0
    for rec in A["images"][:limit]:
        p = os.path.join(img_dir, rec["file_name"])
        if not os.path.exists(p): continue
        im = Image.open(p); im.draft("L", (size, size))
        im = im.convert("L").resize((size, size), Image.BILINEAR)
        mask = Image.new("L", (size, size), 0); d = ImageDraw.Draw(mask)
        sx, sy = size / rec["width"], size / rec["height"]
        for a in by_img.get(rec["id"], []):
            seg = a["segmentation"]
            if isinstance(seg, dict): skipped_rle += 1; continue
            if seg and not isinstance(seg[0], (list, tuple)): seg = [seg]
            for poly in seg:
                pts = np.asarray(poly, dtype=float).reshape(-1, 2)
                if len(pts) >= 3:
                    d.polygon([(x * sx, y * sy) for x, y in pts], fill=255, outline=255)
        a_ = np.asarray(im, dtype=np.float32)
        a_ = (a_ - a_.min()) / (a_.max() - a_.min() + 1e-8)
        X.append(a_); Y.append((np.asarray(mask) > 0).astype(np.uint8)); names.append(os.path.splitext(rec["file_name"])[0])
    print(f"Loaded {len(X)} images ({len(A['images'])} in JSON); RLE annotations skipped: {skipped_rle}")
    return np.stack(X), np.stack(Y), names

X_all, Y_all, names = load_pairs(ANN_PATH, IMG_DIR)
print("images:", X_all.shape, " masks:", Y_all.shape)
print(f"Filament pixel fraction: {Y_all.mean():.4%}")
print(f"Images with no mask pixels: {(Y_all.reshape(len(Y_all), -1).sum(1) == 0).sum()}")

## 5 - Split by day
Whole days go to one split because frames from the same day are near-duplicates. Same procedure and seed as the logistic regression notebook.

In [ ]:
n = len(X_all)
days = np.array([nm[:8] for nm in names])
if SPLIT_BY_DAY and all(d.isdigit() for d in days):
    ud = rng.permutation(np.unique(days))
    cum = np.cumsum([(days == d).sum() for d in ud])
    k1, k2 = np.searchsorted(cum, SPLIT[0] * n) + 1, np.searchsorted(cum, (SPLIT[0] + SPLIT[1]) * n) + 1
    which = {d: 0 for d in ud[:k1]} | {d: 1 for d in ud[k1:k2]} | {d: 2 for d in ud[k2:]}
    grp = np.array([which[d] for d in days])
    idx_tr, idx_va, idx_te = (np.flatnonzero(grp == g) for g in range(3))
else:
    perm = rng.permutation(n)
    n_tr, n_va = int(SPLIT[0] * n), int(SPLIT[1] * n)
    idx_tr, idx_va, idx_te = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
print(f"train={len(idx_tr)}  val={len(idx_va)}  test={len(idx_te)} images")

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j, i in enumerate(idx_tr[:4]):
    ax[0, j].imshow(X_all[i], cmap="gray"); ax[0, j].set_title(names[i]); ax[0, j].axis("off")
    ax[1, j].imshow(Y_all[i], cmap="gray"); ax[1, j].axis("off")
plt.show()

## Dataset statistics and leakage checks
Per-split counts computed from the split indices above (no masks are recomputed), followed by two leakage assertions.

In [ ]:
# Dataset statistics and leakage checks (uses names, days, Y_all, A and idx_tr / idx_va / idx_te from above)
# image records in the same order as load_pairs() (it skips records whose file is missing); some file names repeat
_recs = [r for r in A["images"] if os.path.exists(os.path.join(IMG_DIR, r["file_name"]))]
assert len(_recs) == len(names), (len(_recs), len(names))
_ann_per_img = Counter(a["image_id"] for a in A["annotations"]
                       if CATEGORY_IDS is None or a["category_id"] in CATEGORY_IDS)
_fmt_day = lambda d: f"{d[:4]}-{d[4:6]}-{d[6:8]}" if d.isdigit() else d
split_days, split_names = {}, {}
for split_name, idx in (("train", idx_tr), ("val", idx_va), ("test", idx_te)):
    d = sorted(set(days[idx]))
    split_days[split_name], split_names[split_name] = set(d), {names[i] for i in idx}
    n_px = len(idx) * Y_all[0].size
    frac = sum(int(Y_all[i].sum()) for i in idx) / n_px if n_px else float("nan")
    n_ann = sum(_ann_per_img.get(_recs[i]["id"], 0) for i in idx)
    print(f"{split_name:5s}: images={len(idx):5d}  days={len(d):4d}  "
          f"first={_fmt_day(d[0]) if d else '-'}  last={_fmt_day(d[-1]) if d else '-'}  "
          f"filament pixel fraction={frac:.4%}  annotations={n_ann}")
_name_counts = Counter(names)
print(f"annotations total over splits = {sum(_ann_per_img.get(r['id'], 0) for r in _recs)}  "
      f"(selected annotations in JSON = {sum(_ann_per_img.values())})")
print(f"file names used by more than one image record: {sum(c > 1 for c in _name_counts.values())} "
      f"({sum(c - 1 for c in _name_counts.values())} extra records)")

_pairs = [("train", "val"), ("train", "test"), ("val", "test")]
day_overlap = {f"{a}/{b}": sorted(split_days[a] & split_days[b]) for a, b in _pairs}
name_overlap = {f"{a}/{b}": sorted(split_names[a] & split_names[b]) for a, b in _pairs}
n_day_overlap = sum(len(v) for v in day_overlap.values())
n_name_overlap = sum(len(v) for v in name_overlap.values())
print(f"Check 1 - days in more than one split: {n_day_overlap}", {k: len(v) for k, v in day_overlap.items()})
print(f"Check 2 - file names in more than one split: {n_name_overlap}", {k: len(v) for k, v in name_overlap.items()})
assert n_name_overlap == 0, f"file names shared between splits: {name_overlap}"
if SPLIT_BY_DAY:
    assert n_day_overlap == 0, f"days shared between splits (day split did not run?): {day_overlap}"
else:
    print("SPLIT_BY_DAY is False, so check 1 is reported but not asserted")
print("Leakage checks passed")

## 6 - Dataset and augmentation
Training images get random flips, 90-degree rotations, and a small brightness/contrast change (applied to the image only,
never the mask). Validation and test images are left untouched.

In [ ]:
class SegData(Dataset):
    def __init__(self, X, Y, idxs, augment=False):
        self.X, self.Y, self.idxs, self.augment = X, Y, idxs, augment
    def __len__(self): return len(self.idxs)
    def __getitem__(self, k):
        i = self.idxs[k]
        x = torch.from_numpy(self.X[i])[None]            # (1, H, W)
        y = torch.from_numpy(self.Y[i].astype(np.float32))[None]
        if self.augment:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5: x, y = x.flip(-2), y.flip(-2)
            r = random.randint(0, 3)
            if r: x, y = torch.rot90(x, r, (-2, -1)), torch.rot90(y, r, (-2, -1))
            x = (x * random.uniform(0.85, 1.15) + random.uniform(-0.05, 0.05)).clamp(0, 1)
        return x.contiguous(), y.contiguous()

train_dl = DataLoader(SegData(X_all, Y_all, idx_tr, True), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=len(idx_tr) > BATCH)
val_dl = DataLoader(SegData(X_all, Y_all, idx_va), batch_size=BATCH, num_workers=2)
test_dl = DataLoader(SegData(X_all, Y_all, idx_te), batch_size=BATCH, num_workers=2)
print("batches per epoch:", len(train_dl))

## 7 - Model
A U-Net with a ResNet34 encoder. One input channel (grayscale), one output channel (mask logit). With
`ENCODER_WEIGHTS="imagenet"` the encoder starts from pretrained weights, which usually trains faster and better on small data.
Set it to `None` to train from scratch.

In [ ]:
model = smp.Unet(encoder_name=ENCODER, encoder_weights=ENCODER_WEIGHTS, in_channels=1, classes=1).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8 - Loss
Filaments are a tiny fraction of the pixels, so plain cross-entropy would learn to predict empty masks. The loss is
binary cross-entropy (with the positive class up-weighted by the square root of the imbalance) plus soft Dice loss
computed over the whole batch (so images with no filaments don't cause divide-by-zero).

In [ ]:
pos_frac = float(Y_all[idx_tr].mean())
pos_weight = torch.tensor(float(np.sqrt((1 - pos_frac) / max(pos_frac, 1e-6))), device=device)
print(f"train filament fraction {pos_frac:.4%} -> pos_weight {pos_weight.item():.1f}")
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    return 1 - (2 * (p * y).sum() + eps) / (p.sum() + y.sum() + eps)

def loss_fn(logits, y):
    return bce(logits, y) + dice_loss(logits, y)

## 9 - Metrics
Same pooled IoU / Dice / precision / recall as before, computed over all pixels of the held-out images.

In [ ]:
def confusion(pred, true):
    pred, true = pred.astype(bool), true.astype(bool)
    return (np.sum(pred & true), np.sum(pred & ~true), np.sum(~pred & true))

def scores(tp, fp, fn, eps=1e-8):
    return {"IoU": tp / (tp + fp + fn + eps), "Dice": 2 * tp / (2 * tp + fp + fn + eps),
            "Precision": tp / (tp + fp + eps), "Recall": tp / (tp + fn + eps)}

@torch.no_grad()
def predict_probs(dl):
    model.eval(); out = []
    for x, _ in dl:
        out.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu().numpy())
    return np.concatenate(out)

def evaluate(probs, Y, idxs, thr):
    tp = fp = fn = 0
    for p, i in zip(probs, idxs):
        a, b_, c = confusion(p >= thr, Y[i]); tp += a; fp += b_; fn += c
    return scores(tp, fp, fn)

## 10 - Train
Adam optimizer, mixed precision on GPU. After every epoch the model is scored on the validation images (Dice at threshold 0.5)
and the best checkpoint is kept.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
hist = {"train_loss": [], "val_dice": []}
best_dice, best_state = -1.0, None

for ep in range(EPOCHS):
    model.train(); t0 = time.time(); run = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast(device_type=device, enabled=use_amp):
            logits = model(x)
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run += loss.item() * len(x)
    tr_loss = run / len(train_dl.dataset)
    vd = evaluate(predict_probs(val_dl), Y_all, idx_va, 0.5)["Dice"]
    hist["train_loss"].append(tr_loss); hist["val_dice"].append(float(vd))
    if vd > best_dice:
        best_dice = float(vd); best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    print(f"epoch {ep+1:2d}/{EPOCHS}  train loss {tr_loss:.4f}  val Dice {vd:.4f}  best {best_dice:.4f}  ({time.time()-t0:.0f}s)")

model.load_state_dict(best_state)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist["train_loss"]); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["val_dice"]); ax[1].set_title("validation Dice (thr 0.5)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 11 - Threshold tuning and test results
The threshold is tuned on the validation images only. The test images are scored once.

In [ ]:
val_probs = predict_probs(val_dl)
thr_grid = np.linspace(0.05, 0.95, 19)
best_thr = max(thr_grid, key=lambda t: evaluate(val_probs, Y_all, idx_va, t)["Dice"])
test_probs = predict_probs(test_dl)
unet_val = evaluate(val_probs, Y_all, idx_va, best_thr)
unet_test = evaluate(test_probs, Y_all, idx_te, best_thr)
print(f"Best threshold on val: {best_thr:.2f}")
print("U-Net val :", {k: round(float(v), 4) for k, v in unet_val.items()})
print("U-Net test:", {k: round(float(v), 4) for k, v in unet_test.items()})

## 12 - Comparison table
The first two rows are the numbers from the earlier logistic regression notebook (256 x 256, per-pixel). Edit them if you
re-ran that notebook with different settings. Resolution differs, so treat the comparison as approximate.

**Note on this table:** the "Threshold baseline" and "Logistic regression" rows are copied from `01_logreg_baseline.ipynb`, which runs at **256 x 256**; the U-Net row is computed in this notebook at **512 x 512** (`IMG_SIZE`). The resolutions differ. If `outputs/results_baselines.json` (written by notebook 01) exists, the baseline rows are read from it; otherwise the hardcoded values below are used.

In [ ]:
import pandas as pd
earlier = {
    "Threshold baseline (earlier notebook)":  {"IoU": 0.0039, "Dice": 0.0078, "Precision": 0.0040, "Recall": 0.4082},
    "Logistic regression (earlier notebook)": {"IoU": 0.0138, "Dice": 0.0271, "Precision": 0.0139, "Recall": 0.5848},
}
_bl_path = os.path.join(os.environ.get("OUTPUT_DIR", "outputs"), "results_baselines.json")
if os.path.exists(_bl_path):
    _bl = json.load(open(_bl_path))
    earlier = {"Threshold baseline (earlier notebook)": _bl["Threshold baseline"],
               "Logistic regression (earlier notebook)": _bl["Logistic regression (per-pixel)"]}
    print(f"baseline rows read from {_bl_path} (img_size={_bl['img_size']}, split={_bl['split']})")
else:
    print(f"{_bl_path} not found: baseline rows are the hardcoded values from 01_logreg_baseline.ipynb (256 x 256, test split)")
res = pd.DataFrame({**earlier, f"U-Net {ENCODER}": {k: float(v) for k, v in unet_test.items()}}).T.round(4)
display(res)

## 13 - Visualize predictions
Image, ground truth, U-Net probability, and the thresholded mask. Look for missed thin threads and for dark non-filament regions flagged as filament.

In [ ]:
k = min(4, len(idx_te))
fig, ax = plt.subplots(k, 4, figsize=(14, 3.5 * k), squeeze=False)
for r in range(k):
    i = idx_te[r]
    ax[r, 0].imshow(X_all[i], cmap="gray"); ax[r, 0].set_title(names[i])
    ax[r, 1].imshow(Y_all[i], cmap="gray"); ax[r, 1].set_title("ground truth")
    ax[r, 2].imshow(test_probs[r], cmap="viridis", vmin=0, vmax=1); ax[r, 2].set_title("U-Net probability")
    ax[r, 3].imshow(test_probs[r] >= best_thr, cmap="gray"); ax[r, 3].set_title(f"U-Net mask (thr={best_thr:.2f})")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()

## Next steps
- Compare `ENCODER_WEIGHTS="imagenet"` against `None` (from scratch) for your write-up.
- Try `IMG_SIZE = 256` vs `512` (thin filaments may need more resolution) and more epochs.
- Try other loss weightings (for example, a larger `pos_weight`, or Focal loss).
- Look at the worst test images in section 13 to find systematic failure modes for the report.
- For a Kaggle submission, check the competition's Evaluation page for the required output format before encoding test predictions.

## Planned work (TEAM TO CONFIRM)
- Full-resolution / crop-based training (requested by the professor)
- A second architecture (requested by the professor)
- A hyperparameter comparison (requested by the professor)
- A disc mask for the darkness-threshold baseline